In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
import hashlib

VITORIA

In [5]:

# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

base_url = "https://www.booking.com/reviews/es/hotel/libere-vitoria-centro.es.html"

all_reviews = []
page = 1

while True:
    url = f"{base_url}?aid=356980&label=gog235jc-1DCA0oRkIVbGliZXJlLXZpdG9yaWEtY2VudHJvSDNYA2hGiAEBmAEKuAEXyAEM2AED6AEBiAIBqAIDuALSzuu6BsACAdICJDBiNGMxOWY2LWVlOGMtNDU3ZS05ZDZlLTVmYzM2NDNjZTM5M9gCBOACAQ&sid=b89a34b9ace8403db8dbb71ddde917f9&customer_type=total&hp_nav=0&keep_landing=1&order=featuredreviews&page={page}&r_lang=es&rows=75&"

    print(f"Descargando página {page} ...")
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")  # selector para este link funciona
    if not reviews:
        print("No hay más reseñas, finalizando.")
        break

    print(f"Encontradas {len(reviews)} reseñas en la página {page}")

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(
    os.path.join("Datos", "Transformados", "Vitoria_booking_reviews_todas.csv"),
    index=False,
    encoding="utf-8-sig"
)
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")


Descargando página 1 ...
Encontradas 24 reseñas en la página 1
Descargando página 2 ...
Encontradas 25 reseñas en la página 2
Descargando página 3 ...
Encontradas 25 reseñas en la página 3
Descargando página 4 ...
Encontradas 25 reseñas en la página 4
Descargando página 5 ...
Encontradas 25 reseñas en la página 5
Descargando página 6 ...
Encontradas 25 reseñas en la página 6
Descargando página 7 ...
Encontradas 25 reseñas en la página 7
Descargando página 8 ...
Encontradas 25 reseñas en la página 8
Descargando página 9 ...
Encontradas 25 reseñas en la página 9
Descargando página 10 ...
Encontradas 25 reseñas en la página 10
Descargando página 11 ...
Encontradas 25 reseñas en la página 11
Descargando página 12 ...
Encontradas 25 reseñas en la página 12
Descargando página 13 ...
Encontradas 25 reseñas en la página 13
Descargando página 14 ...
Encontradas 25 reseñas en la página 14
Descargando página 15 ...
Encontradas 25 reseñas en la página 15
Descargando página 16 ...
Encontradas 25 re

In [ ]:
import pandas as pd

df = pd.read_csv("Vitoria_booking_reviews_todas.csv", encoding="utf-8-sig")

# Revisar duplicados por usuario + fecha
duplicates = df.duplicated(subset=["username", "date_written"])
print(f"Número de reseñas duplicadas: {duplicates.sum()}")


Número de reseñas duplicadas: 16


DONOSTI

In [12]:

# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Link del hotel Koisi Hostel en Donosti
base_url = "https://www.booking.com/reviews/es/hotel/koisi-hostel.es.html?aid=356980&label=gog235jc-1DCA0oRkIMa29pc2ktaG9zdGVsSDNYA2hGiAEBmAEKuAEXyAEM2AED6AEBiAIBqAIDuAKIzuu6BsACAdICJDBkYmJjNGJmLTBiYzYtNDUzNS1hYjM2LWU5N2RlMTFkZDU2N9gCBOACAQ&sid=b89a34b9ace8403db8dbb71ddde917f9&customer_type=total&hp_nav=0&keep_landing=1&order=featuredreviews&page=1&r_lang=es&rows=75&"

all_reviews = []
page = 1
last_reviews_count = 0  # Para evitar bucles infinitos

while True:
    url = base_url.replace("page=1", f"page={page}")  # Paginación automática
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")
    current_count = len(reviews)

    if current_count == 0 or current_count == last_reviews_count:
        print("No hay más reseñas o se repite la página, finalizando.")
        break

    print(f"Encontradas {current_count} reseñas en la página {page}")
    last_reviews_count = current_count

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","Donosti_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")


Descargando página 1 ...
Encontradas 24 reseñas en la página 1
Descargando página 2 ...
Encontradas 25 reseñas en la página 2
Descargando página 3 ...
No hay más reseñas o se repite la página, finalizando.
TOTAL RESEÑAS DESCARGADAS: 49


BILBAO MUSEO

In [15]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# URL base del hotel BilbaoMuseo con parámetro rows para obtener más reseñas por página
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-bilbao-guggenheim.es.html?rows=75"

all_reviews = []
seen_reviews = set()  # Para evitar duplicados
page = 1

while True:
    url = f"{base_url}&page={page}"
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    reviews = soup.select("li.review_item")
    if not reviews:
        print("No hay más reseñas visibles, finalizando.")
        break

    new_reviews = 0

    for r_elem in reviews:
        review_text = r_elem.get_text(strip=True)
        review_hash = hashlib.md5(review_text.encode()).hexdigest()

        if review_hash in seen_reviews:
            continue
        seen_reviews.add(review_hash)
        new_reviews += 1

        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    print(f"Página {page} → nuevas reseñas: {new_reviews}")

    if new_reviews == 0:
        print("No se encontraron reseñas nuevas, fin del hotel.")
        break

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","BilbaoMuseo_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"\nTOTAL RESEÑAS DESCARGADAS: {len(df)}")


Descargando página 1 ...
Página 1 → nuevas reseñas: 24
Descargando página 2 ...
Página 2 → nuevas reseñas: 24
Descargando página 3 ...
Página 3 → nuevas reseñas: 25
Descargando página 4 ...
Página 4 → nuevas reseñas: 25
Descargando página 5 ...
No hay más reseñas visibles, finalizando.

TOTAL RESEÑAS DESCARGADAS: 98


bilbao lavieja

In [16]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Link del hotel Koisi Hostel en Donosti
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-bilbao-la-vieja.es.html"

all_reviews = []
page = 1
last_reviews_count = 0  # Para evitar bucles infinitos

while True:
    url = base_url.replace("page=1", f"page={page}")  # Paginación automática
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")
    current_count = len(reviews)

    if current_count == 0 or current_count == last_reviews_count:
        print("No hay más reseñas o se repite la página, finalizando.")
        break

    print(f"Encontradas {current_count} reseñas en la página {page}")
    last_reviews_count = current_count

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","BilbaolaVieja_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Encontradas 24 reseñas en la página 1
Descargando página 2 ...
No hay más reseñas o se repite la página, finalizando.
TOTAL RESEÑAS DESCARGADAS: 24


In [17]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# URL base del hotel BilbaoMuseo con parámetro rows para obtener más reseñas por página
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-bilbao-la-vieja.es.html?rows=75"

all_reviews = []
seen_reviews = set()  # Para evitar duplicados
page = 1

while True:
    url = f"{base_url}&page={page}"
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    reviews = soup.select("li.review_item")
    if not reviews:
        print("No hay más reseñas visibles, finalizando.")
        break

    new_reviews = 0

    for r_elem in reviews:
        review_text = r_elem.get_text(strip=True)
        review_hash = hashlib.md5(review_text.encode()).hexdigest()

        if review_hash in seen_reviews:
            continue
        seen_reviews.add(review_hash)
        new_reviews += 1

        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    print(f"Página {page} → nuevas reseñas: {new_reviews}")

    if new_reviews == 0:
        print("No se encontraron reseñas nuevas, fin del hotel.")
        break

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","BilbaolaVieja_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"\nTOTAL RESEÑAS DESCARGADAS: {len(df)}")


Descargando página 1 ...
Página 1 → nuevas reseñas: 24
Descargando página 2 ...
Página 2 → nuevas reseñas: 25
Descargando página 3 ...
Página 3 → nuevas reseñas: 25
Descargando página 4 ...
Página 4 → nuevas reseñas: 25
Descargando página 5 ...
Página 5 → nuevas reseñas: 25
Descargando página 6 ...
Página 6 → nuevas reseñas: 25
Descargando página 7 ...
No hay más reseñas visibles, finalizando.

TOTAL RESEÑAS DESCARGADAS: 149


ValenciaAbastos

In [18]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Link del hotel Koisi Hostel en Donosti
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-valencia-abastos.es.html"

all_reviews = []
page = 1
last_reviews_count = 0  # Para evitar bucles infinitos

while True:
    url = base_url.replace("page=1", f"page={page}")  # Paginación automática
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")
    current_count = len(reviews)

    if current_count == 0 or current_count == last_reviews_count:
        print("No hay más reseñas o se repite la página, finalizando.")
        break

    print(f"Encontradas {current_count} reseñas en la página {page}")
    last_reviews_count = current_count

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","ValenciaAbastos_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Encontradas 24 reseñas en la página 1
Descargando página 2 ...
No hay más reseñas o se repite la página, finalizando.
TOTAL RESEÑAS DESCARGADAS: 24


In [19]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# URL base del hotel BilbaoMuseo con parámetro rows para obtener más reseñas por página
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-valencia-abastos.es.html?rows=75"

all_reviews = []
seen_reviews = set()  # Para evitar duplicados
page = 1

while True:
    url = f"{base_url}&page={page}"
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    reviews = soup.select("li.review_item")
    if not reviews:
        print("No hay más reseñas visibles, finalizando.")
        break

    new_reviews = 0

    for r_elem in reviews:
        review_text = r_elem.get_text(strip=True)
        review_hash = hashlib.md5(review_text.encode()).hexdigest()

        if review_hash in seen_reviews:
            continue
        seen_reviews.add(review_hash)
        new_reviews += 1

        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    print(f"Página {page} → nuevas reseñas: {new_reviews}")

    if new_reviews == 0:
        print("No se encontraron reseñas nuevas, fin del hotel.")
        break

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","ValenciaAbastos_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"\nTOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Página 1 → nuevas reseñas: 24
Descargando página 2 ...
Página 2 → nuevas reseñas: 25
Descargando página 3 ...
Página 3 → nuevas reseñas: 24
Descargando página 4 ...
Página 4 → nuevas reseñas: 25
Descargando página 5 ...
Página 5 → nuevas reseñas: 25
Descargando página 6 ...
No hay más reseñas visibles, finalizando.

TOTAL RESEÑAS DESCARGADAS: 123


PamplonaYamaguchi

In [20]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Link del hotel Koisi Hostel en Donosti
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-pamplona-yamaguchi.es.html"

all_reviews = []
page = 1
last_reviews_count = 0  # Para evitar bucles infinitos

while True:
    url = base_url.replace("page=1", f"page={page}")  # Paginación automática
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")
    current_count = len(reviews)

    if current_count == 0 or current_count == last_reviews_count:
        print("No hay más reseñas o se repite la página, finalizando.")
        break

    print(f"Encontradas {current_count} reseñas en la página {page}")
    last_reviews_count = current_count

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","PamplonaYamaguchi_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Encontradas 24 reseñas en la página 1
Descargando página 2 ...
No hay más reseñas o se repite la página, finalizando.
TOTAL RESEÑAS DESCARGADAS: 24


In [21]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# URL base del hotel BilbaoMuseo con parámetro rows para obtener más reseñas por página
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-pamplona-yamaguchi.es.html?rows=75"

all_reviews = []
seen_reviews = set()  # Para evitar duplicados
page = 1

while True:
    url = f"{base_url}&page={page}"
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    reviews = soup.select("li.review_item")
    if not reviews:
        print("No hay más reseñas visibles, finalizando.")
        break

    new_reviews = 0

    for r_elem in reviews:
        review_text = r_elem.get_text(strip=True)
        review_hash = hashlib.md5(review_text.encode()).hexdigest()

        if review_hash in seen_reviews:
            continue
        seen_reviews.add(review_hash)
        new_reviews += 1

        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    print(f"Página {page} → nuevas reseñas: {new_reviews}")

    if new_reviews == 0:
        print("No se encontraron reseñas nuevas, fin del hotel.")
        break

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","PamplonaYamaguchi_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"\nTOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Página 1 → nuevas reseñas: 24
Descargando página 2 ...
Página 2 → nuevas reseñas: 25
Descargando página 3 ...
Página 3 → nuevas reseñas: 25
Descargando página 4 ...
Página 4 → nuevas reseñas: 25
Descargando página 5 ...
Página 5 → nuevas reseñas: 25
Descargando página 6 ...
Página 6 → nuevas reseñas: 25
Descargando página 7 ...
Página 7 → nuevas reseñas: 25
Descargando página 8 ...
Página 8 → nuevas reseñas: 25
Descargando página 9 ...
Página 9 → nuevas reseñas: 25
Descargando página 10 ...
Página 10 → nuevas reseñas: 25
Descargando página 11 ...
Página 11 → nuevas reseñas: 25
Descargando página 12 ...
Página 12 → nuevas reseñas: 25
Descargando página 13 ...
Página 13 → nuevas reseñas: 23
Descargando página 14 ...
No hay más reseñas visibles, finalizando.

TOTAL RESEÑAS DESCARGADAS: 322


ValenciaJardinBotanico

In [ ]:

# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Link del hotel Koisi Hostel en Donosti
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-valencia-jardin-botanico.es.html"

all_reviews = []
page = 1
last_reviews_count = 0  # Para evitar bucles infinitos

while True:
    url = base_url.replace("page=1", f"page={page}")  # Paginación automática
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")
    current_count = len(reviews)

    if current_count == 0 or current_count == last_reviews_count:
        print("No hay más reseñas o se repite la página, finalizando.")
        break

    print(f"Encontradas {current_count} reseñas en la página {page}")
    last_reviews_count = current_count

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","ValenciaJardinBotanico_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Encontradas 24 reseñas en la página 1
Descargando página 2 ...
No hay más reseñas o se repite la página, finalizando.
TOTAL RESEÑAS DESCARGADAS: 24


In [ ]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# URL base del hotel BilbaoMuseo con parámetro rows para obtener más reseñas por página
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-valencia-jardin-botanico.es.html?rows=75"

all_reviews = []
seen_reviews = set()  # Para evitar duplicados
page = 1

while True:
    url = f"{base_url}&page={page}"
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    reviews = soup.select("li.review_item")
    if not reviews:
        print("No hay más reseñas visibles, finalizando.")
        break

    new_reviews = 0

    for r_elem in reviews:
        review_text = r_elem.get_text(strip=True)
        review_hash = hashlib.md5(review_text.encode()).hexdigest()

        if review_hash in seen_reviews:
            continue
        seen_reviews.add(review_hash)
        new_reviews += 1

        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    print(f"Página {page} → nuevas reseñas: {new_reviews}")

    if new_reviews == 0:
        print("No se encontraron reseñas nuevas, fin del hotel.")
        break

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","ValenciaJardinBotanico_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"\nTOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Página 1 → nuevas reseñas: 24
Descargando página 2 ...
Página 2 → nuevas reseñas: 25
Descargando página 3 ...
Página 3 → nuevas reseñas: 25
Descargando página 4 ...
Página 4 → nuevas reseñas: 25
Descargando página 5 ...
No hay más reseñas visibles, finalizando.

TOTAL RESEÑAS DESCARGADAS: 99


MadridPalacioReal

In [ ]:

# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Link del hotel Koisi Hostel en Donosti
base_url = "https://www.booking.com/reviews/es/hotel/libere-madrid-palacio-real.es.html"

all_reviews = []
page = 1
last_reviews_count = 0  # Para evitar bucles infinitos

while True:
    url = base_url.replace("page=1", f"page={page}")  # Paginación automática
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")
    current_count = len(reviews)

    if current_count == 0 or current_count == last_reviews_count:
        print("No hay más reseñas o se repite la página, finalizando.")
        break

    print(f"Encontradas {current_count} reseñas en la página {page}")
    last_reviews_count = current_count

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","MadridPalacioReal_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Encontradas 24 reseñas en la página 1
Descargando página 2 ...
No hay más reseñas o se repite la página, finalizando.
TOTAL RESEÑAS DESCARGADAS: 24


In [6]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# URL base del hotel BilbaoMuseo con parámetro rows para obtener más reseñas por página
base_url = "https://www.booking.com/reviews/es/hotel/libere-madrid-palacio-real.es.html?rows=75"

all_reviews = []
seen_reviews = set()  # Para evitar duplicados
page = 1

while True:
    url = f"{base_url}&page={page}"
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    reviews = soup.select("li.review_item")
    if not reviews:
        print("No hay más reseñas visibles, finalizando.")
        break

    new_reviews = 0

    for r_elem in reviews:
        review_text = r_elem.get_text(strip=True)
        review_hash = hashlib.md5(review_text.encode()).hexdigest()

        if review_hash in seen_reviews:
            continue
        seen_reviews.add(review_hash)
        new_reviews += 1

        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    print(f"Página {page} → nuevas reseñas: {new_reviews}")

    if new_reviews == 0:
        print("No se encontraron reseñas nuevas, fin del hotel.")
        break

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","MadridPalacioReal_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"\nTOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Página 1 → nuevas reseñas: 24
Descargando página 2 ...
Página 2 → nuevas reseñas: 25
Descargando página 3 ...
Página 3 → nuevas reseñas: 25
Descargando página 4 ...
Página 4 → nuevas reseñas: 25
Descargando página 5 ...
Página 5 → nuevas reseñas: 24
Descargando página 6 ...
No hay más reseñas visibles, finalizando.

TOTAL RESEÑAS DESCARGADAS: 123


MalagaTeatroRomano

In [ ]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Link del hotel Koisi Hostel en Donosti
base_url = "https://www.booking.com/reviews/es/hotel/apartamentosliberemalagateatroromano.es.html"

all_reviews = []
page = 1
last_reviews_count = 0  # Para evitar bucles infinitos

while True:
    url = base_url.replace("page=1", f"page={page}")  # Paginación automática
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")
    current_count = len(reviews)

    if current_count == 0 or current_count == last_reviews_count:
        print("No hay más reseñas o se repite la página, finalizando.")
        break

    print(f"Encontradas {current_count} reseñas en la página {page}")
    last_reviews_count = current_count

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","MalagaTeatroRomano_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Encontradas 24 reseñas en la página 1
Descargando página 2 ...
No hay más reseñas o se repite la página, finalizando.
TOTAL RESEÑAS DESCARGADAS: 24


In [ ]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# URL base del hotel BilbaoMuseo con parámetro rows para obtener más reseñas por página
base_url = "https://www.booking.com/reviews/es/hotel/apartamentosliberemalagateatroromano.es.html?rows=75"

all_reviews = []
seen_reviews = set()  # Para evitar duplicados
page = 1

while True:
    url = f"{base_url}&page={page}"
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    reviews = soup.select("li.review_item")
    if not reviews:
        print("No hay más reseñas visibles, finalizando.")
        break

    new_reviews = 0

    for r_elem in reviews:
        review_text = r_elem.get_text(strip=True)
        review_hash = hashlib.md5(review_text.encode()).hexdigest()

        if review_hash in seen_reviews:
            continue
        seen_reviews.add(review_hash)
        new_reviews += 1

        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    print(f"Página {page} → nuevas reseñas: {new_reviews}")

    if new_reviews == 0:
        print("No se encontraron reseñas nuevas, fin del hotel.")
        break

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","MalagaTeatroRomano_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"\nTOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Página 1 → nuevas reseñas: 24
Descargando página 2 ...
Página 2 → nuevas reseñas: 25
Descargando página 3 ...
Página 3 → nuevas reseñas: 3
Descargando página 4 ...
No hay más reseñas visibles, finalizando.

TOTAL RESEÑAS DESCARGADAS: 52


GranadaCatedral

In [ ]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Link del hotel Koisi Hostel en Donosti
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-granada-catedral.es.html"

all_reviews = []
page = 1
last_reviews_count = 0  # Para evitar bucles infinitos

while True:
    url = base_url.replace("page=1", f"page={page}")  # Paginación automática
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")
    current_count = len(reviews)

    if current_count == 0 or current_count == last_reviews_count:
        print("No hay más reseñas o se repite la página, finalizando.")
        break

    print(f"Encontradas {current_count} reseñas en la página {page}")
    last_reviews_count = current_count

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","GranadaCatedral_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Encontradas 24 reseñas en la página 1
Descargando página 2 ...
No hay más reseñas o se repite la página, finalizando.
TOTAL RESEÑAS DESCARGADAS: 24


In [ ]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# URL base del hotel BilbaoMuseo con parámetro rows para obtener más reseñas por página
base_url = "https://www.booking.com/reviews/es/hotel/apartamentos-libere-granada-catedral.es.html?rows=75"

all_reviews = []
seen_reviews = set()  # Para evitar duplicados
page = 1

while True:
    url = f"{base_url}&page={page}"
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    reviews = soup.select("li.review_item")
    if not reviews:
        print("No hay más reseñas visibles, finalizando.")
        break

    new_reviews = 0

    for r_elem in reviews:
        review_text = r_elem.get_text(strip=True)
        review_hash = hashlib.md5(review_text.encode()).hexdigest()

        if review_hash in seen_reviews:
            continue
        seen_reviews.add(review_hash)
        new_reviews += 1

        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    print(f"Página {page} → nuevas reseñas: {new_reviews}")

    if new_reviews == 0:
        print("No se encontraron reseñas nuevas, fin del hotel.")
        break

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","GranadaCatedral_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"\nTOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...


NameError: name 'hashlib' is not defined

MalagaLaMerced

In [10]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Link del hotel Koisi Hostel en Donosti
base_url = "https://www.booking.com/reviews/es/hotel/libere-malaga-la-merced.es.html"

all_reviews = []
page = 1
last_reviews_count = 0  # Para evitar bucles infinitos

while True:
    url = base_url.replace("page=1", f"page={page}")  # Paginación automática
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")
    current_count = len(reviews)

    if current_count == 0 or current_count == last_reviews_count:
        print("No hay más reseñas o se repite la página, finalizando.")
        break

    print(f"Encontradas {current_count} reseñas en la página {page}")
    last_reviews_count = current_count

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","MalagaLaMerced_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Encontradas 24 reseñas en la página 1
Descargando página 2 ...
No hay más reseñas o se repite la página, finalizando.
TOTAL RESEÑAS DESCARGADAS: 24


In [9]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# URL base del hotel BilbaoMuseo con parámetro rows para obtener más reseñas por página
base_url = "https://www.booking.com/reviews/es/hotel/libere-malaga-la-merced.es.html?rows=75"

all_reviews = []
seen_reviews = set()  # Para evitar duplicados
page = 1

while True:
    url = f"{base_url}&page={page}"
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    reviews = soup.select("li.review_item")
    if not reviews:
        print("No hay más reseñas visibles, finalizando.")
        break

    new_reviews = 0

    for r_elem in reviews:
        review_text = r_elem.get_text(strip=True)
        review_hash = hashlib.md5(review_text.encode()).hexdigest()

        if review_hash in seen_reviews:
            continue
        seen_reviews.add(review_hash)
        new_reviews += 1

        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    print(f"Página {page} → nuevas reseñas: {new_reviews}")

    if new_reviews == 0:
        print("No se encontraron reseñas nuevas, fin del hotel.")
        break

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","MalagaLaMerced_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"\nTOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...


NameError: name 'hashlib' is not defined

CordobaPatio

In [7]:
# --- Configuración ---
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Link del hotel Koisi Hostel en Donosti
base_url = "https://www.booking.com/reviews/es/hotel/libere-cordoba-patio-santa-marta.es.html?aid=356980&label=gog235jc-1DCA0oRkIgbGliZXJlLWNvcmRvYmEtcGF0aW8tc2FudGEtbWFydGFIM1gDaEaIAQGYAQq4ARfIAQzYAQPoAQGIAgGoAgO4AruB8LoGwAIB0gIkZWRiMTQzODktYmYyZi00NGJkLTg4MDYtNzBlYjc1YmFiZTM12AIE4AIB&sid=b89a34b9ace8403db8dbb71ddde917f9&customer_type=total&hp_nav=0&keep_landing=1&order=featuredreviews&page=1&r_lang=es&rows=75&"

all_reviews = []
page = 1
last_reviews_count = 0  # Para evitar bucles infinitos

while True:
    url = base_url.replace("page=1", f"page={page}")  # Paginación automática
    print(f"Descargando página {page} ...")
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Seleccionar todas las reseñas
    reviews = soup.select("li.review_item")
    current_count = len(reviews)

    if current_count == 0 or current_count == last_reviews_count:
        print("No hay más reseñas o se repite la página, finalizando.")
        break

    print(f"Encontradas {current_count} reseñas en la página {page}")
    last_reviews_count = current_count

    for r_elem in reviews:
        username = r_elem.select_one(".reviewer_name")
        country = r_elem.select_one(".reviewer_country")
        score = r_elem.select_one(".review-score-badge")
        date_written = r_elem.select_one(".review_item_date")
        positive = r_elem.select_one(".review_pos")
        negative = r_elem.select_one(".review_neg")
        tags_elements = r_elem.select(".review_info_tag")
        tags = [t.get_text(strip=True) for t in tags_elements]

        # Convertir puntuación a float
        score_val = None
        if score:
            score_text = score.get_text(strip=True).replace(",", ".")
            try:
                score_val = float(score_text)
            except:
                score_val = None

        all_reviews.append({
            "username": username.get_text(strip=True) if username else None,
            "country": country.get_text(strip=True) if country else None,
            "score": score_val,
            "date_written": date_written.get_text(strip=True) if date_written else None,
            "positive": positive.get_text(strip=True) if positive else None,
            "negative": negative.get_text(strip=True) if negative else None,
            "tags": tags
        })

    page += 1
    time.sleep(random.uniform(2, 4))  # Espera aleatoria

# Guardar CSV
df = pd.DataFrame(all_reviews)
df.to_csv(os.path.join("Datos", "Transformados","CordobaPatio_booking_reviews_todas.csv"), index=False, encoding="utf-8-sig")
print(f"TOTAL RESEÑAS DESCARGADAS: {len(df)}")

Descargando página 1 ...
Encontradas 25 reseñas en la página 1
Descargando página 2 ...
No hay más reseñas o se repite la página, finalizando.
TOTAL RESEÑAS DESCARGADAS: 25
